In [7]:
# Install required libraries
!pip install -q sentence-transformers transformers torch torchvision numpy pillow chromadb openai spacy clip


  Preparing metadata (setup.py) ... done


In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [6]:
import json
import spacy

def semantic_chunking(text, chunk_size=450, overlap=200):
    """
    Splits text into semantically meaningful chunks using sentence boundaries.
    """
    nlp = spacy.load("en_core_web_sm")  # Load small spaCy model for efficient processing
    doc = nlp(text)
    sentences = [sent.text for sent in doc.sents]  # Extract sentences

    chunks = []
    current_chunk = []
    current_length = 0

    for sentence in sentences:
        sentence_length = len(sentence)

        if current_length + sentence_length > chunk_size and current_chunk:
            # Store current chunk
            chunks.append(" ".join(current_chunk))

            # Create overlap with previous chunk
            overlap_sentences = current_chunk[-overlap:] if overlap < len(current_chunk) else current_chunk[:]
            current_chunk = overlap_sentences[:]
            current_length = sum(len(s) for s in current_chunk)

        current_chunk.append(sentence)
        current_length += sentence_length

    # Add the final chunk
    if current_chunk:
        chunks.append(" ".join(current_chunk))

    return chunks

# Load the cleaned JSON file
file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/cleaned_winning_models.json"
with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# Process each entry and apply semantic chunking
final_chunked_data = []
for entry in data:
    model_name = entry.get("model_name", "")
    description = entry.get("description", "")
    system_benefits = entry.get("system_benefits", "")
    image_path = entry.get("image_path", "")
    image_description = entry.get("image_description", "")
    url = entry.get("url", "")
    category = entry.get("category", "")
    additional_metadata = {key: value for key, value in entry.items() if key not in ["model_name", "description", "system_benefits", "image_path", "image_description", "url", "category"]}

    # Concatenating relevant fields
    full_text = f"{model_name}. {description} {system_benefits}"

    # Apply semantic chunking
    text_chunks = semantic_chunking(full_text, chunk_size=450, overlap=200)

    # Create chunked entries while keeping image_path and metadata associated
    for idx, chunk in enumerate(text_chunks):
        chunk_entry = {
            "model_name": model_name,
            "chunk_index": idx,
            "chunk_text": chunk,
            "image_path": image_path,
            "image_description": image_description,
            "url": url,
            "category": category
        }
        chunk_entry.update(additional_metadata)  # Add extra metadata
        final_chunked_data.append(chunk_entry)

# Save the final chunked JSON file
final_chunked_file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/final_semantic_chunked_models.json"
with open(final_chunked_file_path, "w", encoding="utf-8") as file:
    json.dump(final_chunked_data, file, indent=4, ensure_ascii=False)

# Provide the final chunked file path
print(f"Chunked data saved at: {final_chunked_file_path}")



/usr/local/lib/python3.11/dist-packages/spacy/util.py:1740: UserWarning: [W111] Jupyter notebook detected: if using `prefer_gpu()` or `require_gpu()`, include it in the same cell right before `spacy.load()` to ensure that the model is loaded on the correct device. More information: http://spacy.io/usage/v3#jupyter-notebook-gpu
  warnings.warn(Warnings.W111)


Chunked data saved at: /content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/final_semantic_chunked_models.json


In [9]:
import json
import torch
import os
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
from tqdm import tqdm

def process_text_embedding(text_chunks, model, processor, device):
    """Generate CLIP text embeddings in batches to optimize memory usage."""
    embeddings = []
    batch_size = 50  # Process 50 text chunks at a time

    for i in tqdm(range(0, len(text_chunks), batch_size), desc="Processing Text Chunks"):
        batch = text_chunks[i : i + batch_size]
        inputs = processor(text=batch, return_tensors="pt", padding=True, truncation=True).to(device)
        with torch.no_grad():
            text_embeddings = model.get_text_features(**inputs).cpu().numpy().tolist()
        embeddings.extend(text_embeddings)

    return embeddings

def process_image_embedding(image_paths, model, processor, device):
    """Generate CLIP image embeddings in batches while handling missing images."""
    embeddings = []
    batch_size = 10  # Process images in smaller batches to prevent memory overload

    for i in tqdm(range(0, len(image_paths), batch_size), desc="Processing Image Chunks"):
        batch = image_paths[i : i + batch_size]
        image_tensors = []
        valid_indices = []

        for idx, image_path in enumerate(batch):
            if image_path and os.path.exists(image_path):
                try:
                    image = Image.open(image_path).convert("RGB")
                    image_tensors.append(image)
                    valid_indices.append(i + idx)
                except Exception as e:
                    print(f"Error processing image {image_path}: {e}")

        if image_tensors:
            inputs = processor(images=image_tensors, return_tensors="pt").to(device)
            with torch.no_grad():
                image_embeddings = model.get_image_features(**inputs).cpu().numpy().tolist()
            for idx, emb in zip(valid_indices, image_embeddings):
                embeddings.append((idx, emb))

    return embeddings

# Load CLIP model
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Load the final semantic chunked JSON file
file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/final_semantic_chunked_models.json"
with open(file_path, "r", encoding="utf-8") as file:
    data = json.load(file)

# Extract text chunks and image paths
text_chunks = [entry["chunk_text"] for entry in data]
image_paths = [entry["image_path"] for entry in data]

# Generate text embeddings
text_embeddings = process_text_embedding(text_chunks, model, processor, device)

# Generate image embeddings
image_embeddings = process_image_embedding(image_paths, model, processor, device)

# Store embeddings
embeddings = []
for idx, entry in enumerate(data):
    image_embedding = next((emb for img_idx, emb in image_embeddings if img_idx == idx), None)
    embeddings.append({
        "model_name": entry.get("model_name", ""),
        "chunk_index": entry.get("chunk_index", 0),
        "text_embedding": text_embeddings[idx],
        "image_embedding": image_embedding,
        "metadata": entry
    })

# Save embeddings to JSON
embeddings_file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/optimized_clip_embeddings.json"
with open(embeddings_file_path, "w", encoding="utf-8") as file:
    json.dump(embeddings, file, indent=4, ensure_ascii=False)

# Provide the final path
print(f"Embeddings saved at: {embeddings_file_path}")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/4.19k [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/605M [00:00<?, ?B/s]

preprocessor_config.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/605M [00:00<?, ?B/s]

...

merges.txt:   0%|          | 0.00/525k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.22M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/389 [00:00<?, ?B/s]

Processing Image Chunks: 100%|██████████| 26/26 [01:10<00:00,  2.71s/it]


Embeddings saved at: /content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/optimized_clip_embeddings.json


In [44]:
import json
import chromadb
import torch
from transformers import CLIPModel, CLIPProcessor

# Load the CLIP model for query embedding
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Initialize ChromaDB client
chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/chroma_db")
collection = chroma_client.get_or_create_collection(name="clip_embeddings")

# Load CLIP embeddings JSON file
embeddings_file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/optimized_clip_embeddings.json"
with open(embeddings_file_path, "r", encoding="utf-8") as file:
    embeddings_data = json.load(file)

# Fetch stored embeddings from ChromaDB
stored_data = collection.get(include=["metadatas"], limit=1000)  # Adjust limit as needed

# Extract existing IDs from metadata
existing_ids = set()
for metadata in stored_data["metadatas"]:
    if metadata and "model_name" in metadata and "chunk_index" in metadata:
        unique_id = f"{metadata['model_name'].replace(' ', '_')}_{metadata['chunk_index']}"
        existing_ids.add(f"text_{unique_id}")
        existing_ids.add(f"image_{unique_id}")

# Function to ensure correct embedding format
def format_embedding(embedding):
    if isinstance(embedding, list) and len(embedding) > 0 and isinstance(embedding[0], list):
        return embedding[0]  # Flatten nested lists
    return embedding  # Return as is if already correct

# Store embeddings in ChromaDB with unique IDs
for entry in embeddings_data:
    unique_id = f"{entry['metadata']['model_name'].replace(' ', '_')}_{entry['chunk_index']}"
    text_embedding = format_embedding(entry.get("text_embedding"))
    image_embedding = format_embedding(entry.get("image_embedding"))
    metadata = entry.get("metadata", {})

    # Store text embedding only if it's not already in ChromaDB
    if f"text_{unique_id}" not in existing_ids and text_embedding:
        collection.add(
            ids=[f"text_{unique_id}"],
            embeddings=[text_embedding],
            metadatas=[metadata]
        )

    # Store image embedding only if it's not already in ChromaDB
    if f"image_{unique_id}" not in existing_ids and image_embedding:
        collection.add(
            ids=[f"image_{unique_id}"],
            embeddings=[image_embedding],
            metadatas=[metadata]
        )

print("✅ Text and image embeddings successfully stored in ChromaDB.")


✅ Text and image embeddings successfully stored in ChromaDB.


In [53]:
import os
import json
import torch
from PIL import Image
from transformers import CLIPModel, CLIPProcessor
import chromadb

# Reconnect to ChromaDB
chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/chroma_db")
collection = chroma_client.get_or_create_collection(name="clip_embeddings")

# Load CLIP model for text and image processing
device = "cuda" if torch.cuda.is_available() else "cpu"
model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device)
processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Load JSON data with image paths
embeddings_file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/optimized_clip_embeddings.json"
with open(embeddings_file_path, "r", encoding="utf-8") as file:
    embeddings_data = json.load(file)

# Store image embeddings
for entry in embeddings_data:
    image_path = entry.get("image_path", None)
    model_name = entry["metadata"]["model_name"].replace(" ", "_")
    chunk_index = entry["chunk_index"]

    # Skip if image_path is missing
    if not image_path or not os.path.exists(image_path):
        continue

    # Load image and process
    image = Image.open(image_path).convert("RGB")
    inputs = processor(images=image, return_tensors="pt").to(device)

    with torch.no_grad():
        image_embedding = model.get_image_features(**inputs).detach().cpu().numpy().tolist()[0]

    # Store in ChromaDB
    collection.add(
        ids=[f"image_{model_name}_{chunk_index}"],
        embeddings=[image_embedding],
        metadatas=[entry["metadata"]]
    )

print("✅ Image embeddings successfully stored in ChromaDB.")


✅ Image embeddings successfully stored in ChromaDB.


In [55]:
# ✅ Verify if image embeddings are correctly retrieved
stored_data = collection.get(include=["metadatas", "embeddings"], limit=10000)

# Count image embeddings
stored_image_count = sum(1 for meta in stored_data["metadatas"] if "image_path" in meta)
print(f"✅ Retrieved Image Embeddings Count: {stored_image_count}")

✅ Retrieved Image Embeddings Count: 540


In [3]:
!pip install ChromaDB

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 67.3/67.3 kB 2.8 MB/s eta 0:00:00
  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 611.1/611.1 kB 13.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.4/2.4 MB 46.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.6/278.6 kB 16.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 94.8/94.8 kB 6.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 46.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 101.6/101.6 kB 7.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.3/13.3 MB 37.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 55.9/55.9 kB 3.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 177.4/177.4 kB 10.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.0/65.0 kB 3.8 MB/s eta 0:0

#Fresh on ChromaDB

In [2]:
#Text Embedding
import json

# Load chunked JSON data
chunked_file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/final_semantic_chunked_models.json"

with open(chunked_file_path, "r", encoding="utf-8") as file:
    chunked_data = json.load(file)

from sentence_transformers import SentenceTransformer

# Load a lightweight text embedding model
text_model = SentenceTransformer("all-MiniLM-L6-v2")

# Generate embeddings for each chunk of text
for entry in chunked_data:
    entry["text_embedding"] = text_model.encode(entry["chunk_text"]).tolist()

print("Text embeddings generated successfully!")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.7k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

...

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

1_Pooling/config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

Text embeddings generated successfully!


In [1]:
import json
import hashlib
import chromadb
from sentence_transformers import SentenceTransformer

# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/End/chromadb_store")


# Load chunked JSON data
chunked_file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/final_semantic_chunked_models.json"

with open(chunked_file_path, "r", encoding="utf-8") as file:
    chunked_data = json.load(file)

# Initialize sentence-transformers model for text embeddings
text_model = SentenceTransformer("all-MiniLM-L6-v2")


# Create a collection for text embeddings
text_collection = chroma_client.get_or_create_collection(name="rag_text_data")

# Function to generate a unique ID while keeping chunk_index separate
def generate_unique_id(model_name, chunk_index, chunk_text):
    """Generate a unique ID using model name, chunk index, and text hash."""
    text_hash = hashlib.md5(chunk_text.encode()).hexdigest()  # Hash the chunk text
    return f"{model_name.replace(' ', '_')}_{text_hash}"  # Unique ID

# Store text embeddings while keeping chunk_index correct
for entry in chunked_data:
    unique_id = generate_unique_id(entry["model_name"], entry["chunk_index"], entry["chunk_text"])

    # Generate text embedding
    text_embedding = text_model.encode(entry["chunk_text"]).tolist()

    # Store in ChromaDB
    text_collection.add(
        ids=[unique_id],  # Unique ID
        embeddings=[text_embedding],  # Store the vector embedding
        metadatas=[{
            "model_name": entry["model_name"],
            "chunk_index": entry["chunk_index"],  # Keep chunk_index for ordering
            "chunk_text": entry["chunk_text"],
            "category": entry["category"],
            "url": entry["url"]
        }]
    )

print("Text embeddings successfully stored in ChromaDB!")


/usr/local/lib/python3.11/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


Text embeddings successfully stored in ChromaDB!


In [2]:
import json
import hashlib
import chromadb
from sentence_transformers import SentenceTransformer

# Load chunked JSON data
chunked_file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/final_semantic_chunked_models.json"

with open(chunked_file_path, "r", encoding="utf-8") as file:
    chunked_data = json.load(file)

# Initialize sentence-transformers model for text embeddings
text_model = SentenceTransformer("all-MiniLM-L6-v2")

# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/End/chromadb_store")

# Create a collection for text embeddings
text_collection = chroma_client.get_or_create_collection(name="rag_text_data")

# Track unique IDs
seen_ids = set()

def generate_unique_id(model_name, chunk_index, chunk_text):
    """Generate a unique ID using model name, chunk index, and text hash."""
    text_hash = hashlib.md5(chunk_text.encode()).hexdigest()  # Hash the chunk text
    return f"{model_name.replace(' ', '_')}_{text_hash}"  # Unique ID

# Track counts for validation
total_chunks = len(chunked_data)
stored_chunks = 0

# Store text embeddings while tracking count
for entry in chunked_data:
    unique_id = generate_unique_id(entry["model_name"], entry["chunk_index"], entry["chunk_text"])

    # Ensure the unique ID is not duplicated
    if unique_id in seen_ids:
        print(f"Skipping duplicate entry: {entry['model_name']} (Chunk {entry['chunk_index']})")
        continue  # Skip duplicates

    # Mark as seen
    seen_ids.add(unique_id)

    # Generate text embedding
    text_embedding = text_model.encode(entry["chunk_text"]).tolist()

    # Store in ChromaDB
    text_collection.add(
        ids=[unique_id],
        embeddings=[text_embedding],
        metadatas=[{
            "model_name": entry["model_name"],
            "chunk_index": entry["chunk_index"],
            "chunk_text": entry["chunk_text"],
            "category": entry["category"],
            "url": entry["url"]
        }]
    )

    stored_chunks += 1

# Validate stored count in ChromaDB
db_count = len(text_collection.get()['ids'])

print("\n--- Validation Summary ---")
print(f"Total Chunks Processed: {total_chunks}")
print(f"Total Unique Chunks Stored: {stored_chunks}")
print(f"Total Records in ChromaDB: {db_count}")

# Check for mismatches
if stored_chunks == db_count:
    print("✅ Data validation successful: ChromaDB contains expected embeddings.")
else:
    print("⚠️ Mismatch detected! Check for missing or duplicated entries.")



--- Validation Summary ---
Total Chunks Processed: 257
Total Unique Chunks Stored: 257
Total Records in ChromaDB: 257
✅ Data validation successful: ChromaDB contains expected embeddings.


In [3]:
query_text = "How does ADAS improve safety in autonomous driving?"
query_embedding = text_model.encode(query_text).tolist()

# Perform a text search
results = text_collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

# Display results
for result in results["metadatas"][0]:
    print(f"Model: {result['model_name']}, Chunk Index: {result['chunk_index']}\nChunk Text: {result['chunk_text']}\nURL: {result['url']}\n")


Model: Comprehensive ADAS & Autonomous Driving Hardware-Software System, Chunk Index: 1
Chunk Text: Comprehensive ADAS & Autonomous Driving Hardware-Software System. This is about application Comprehensive ADAS & Autonomous Driving Hardware-Software System: The automotive industry is rapidly advancing towards autonomous driving and enhanced safety features, such as advanced driver assistance systems (ADAS) with technologies like lane departure warnings, collision avoidance, and driver monitoring systems. The global push for safer vehicles, coupled with regulatory pressures and consumer expectations, is driving the adoption of sophisticated ADAS solutions.
URL: https://www.renesas.com/en/applications/automotive/adas/comprehensive-adas-autonomous-driving-hardware-software-system

Model: Comprehensive ADAS & Autonomous Driving Hardware-Software System, Chunk Index: 0
Chunk Text: Comprehensive ADAS & Autonomous Driving Hardware-Software System. This is about application Comprehensive ADAS 

In [4]:
query_text = "what are the system benefits of Wireless EV Battery Management System?"
query_embedding = text_model.encode(query_text).tolist()

# Perform a text search
results = text_collection.query(
    query_embeddings=[query_embedding],
    n_results=3
)

# Display results
for result in results["metadatas"][0]:
    print(f"Model: {result['model_name']}, Chunk Index: {result['chunk_index']}\nChunk Text: {result['chunk_text']}\nURL: {result['url']}\n")

Model: Wireless EV Battery Management System, Chunk Index: 1
Chunk Text: Wireless EV Battery Management System. This is about application Wireless EV Battery Management System: Renesas' automotive wireless battery management system (BMS) eliminates wire harnesses allowing for flexible battery placement, simplifying the development of scalable electric vehicles. The system benefits for application Wireless EV Battery Management System are:  Eliminates the traditional wire harnesses required in a BMS, saving weight and space while improving flexibility Easier battery replacement and reuse throughout the life cycle Renesas’ low power Bluetooth® Low Energy (LE) 5.1 device*1 for wireless BMS is ideal for battery life cycle standardization based on the open Bluetooth LE standard, instead of a proprietary wireless protocol unique to a single supplier Single-cell/Bluetooth LE with a Bluetooth LE MCU module for a flexible cell attachment/detachment concept A system-level functional safety conce

##Text Embedding

In [5]:
!pip install transformers torch pillow


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 88.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 70.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 1.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 14.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 7.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 75.2 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninstalling nvidia-nvjitlink-cu12-12.5.82:
      Successfully uninstalled nvidia-nvjitlin

In [7]:
import json
import hashlib
import chromadb
import torch
from sentence_transformers import SentenceTransformer
from transformers import CLIPProcessor, CLIPModel
from PIL import Image

# Load chunked JSON data
chunked_file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/final_semantic_chunked_models.json"

with open(chunked_file_path, "r", encoding="utf-8") as file:
    chunked_data = json.load(file)

# Load text embedding model
text_model = SentenceTransformer("all-MiniLM-L6-v2")

# Load CLIP model for image + text embeddings
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32")
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

# Initialize ChromaDB
chroma_client = chromadb.PersistentClient(path="/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/End/chromadb_store")

# Create a collection for combined image + text embeddings
image_text_collection = chroma_client.get_or_create_collection(name="rag_image_text_data")

def generate_unique_id(model_name, chunk_index, chunk_text):
    """Generate a unique ID using model name, chunk index, and text hash."""
    text_hash = hashlib.md5(chunk_text.encode()).hexdigest()
    return f"{model_name.replace(' ', '_')}_{text_hash}"

def get_combined_embedding(image_path, image_description, chunk_text):
    """Generate a multimodal embedding by combining image, description, and text."""
    try:
        # Get text embedding (chunk_text + image_description)
        combined_text = f"{chunk_text} {image_description}"
        text_embedding = text_model.encode(combined_text).tolist()

        # Process image separately
        image = Image.open(image_path).convert("RGB")
        image_inputs = clip_processor(images=image, return_tensors="pt")
        text_inputs = clip_processor(text=[image_description], return_tensors="pt", padding=True, truncation=True)

        with torch.no_grad():
            img_embedding = clip_model.get_image_features(**image_inputs).squeeze().tolist()
            clip_text_embedding = clip_model.get_text_features(**text_inputs).squeeze().tolist()

        # Combine embeddings (weighted average)
        final_embedding = [(t + i + c) / 3 for t, i, c in zip(text_embedding, img_embedding, clip_text_embedding)]
        return final_embedding

    except Exception as e:
        print(f"Error processing image: {image_path}, Error: {e}")
        return None  # Handle missing or invalid images

# Store Image + Text Embeddings
stored_images = 0
for entry in chunked_data:
    unique_id = generate_unique_id(entry["model_name"], entry["chunk_index"], entry["chunk_text"])

    if entry["image_path"]:
        combined_embedding = get_combined_embedding(entry["image_path"], entry["image_description"], entry["chunk_text"])

        if combined_embedding:
            # Store in ChromaDB
            image_text_collection.add(
                ids=[unique_id + "_img"],
                embeddings=[combined_embedding],
                metadatas=[{
                    "model_name": entry["model_name"],
                    "image_path": entry["image_path"],
                    "image_description": entry["image_description"],
                    "chunk_text": entry["chunk_text"],
                    "category": entry["category"],
                    "url": entry["url"]
                }]
            )
            stored_images += 1

print(f"✅ Successfully stored {stored_images} image + text embeddings in ChromaDB!")


✅ Successfully stored 257 image + text embeddings in ChromaDB!


In [8]:
query_text = "Show me a diagram related to ADAS safety."
query_embedding = text_model.encode(query_text).tolist()

# Perform similarity search
results = image_text_collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

# Display results
print("\n--- Image + Text Search Results ---")
for result in results["metadatas"][0]:
    print(f"🔹 Model: {result['model_name']}")
    print(f"📖 Chunk Text: {result['chunk_text']}")
    print(f"🖼 Image Path: {result['image_path']}")
    print(f"📝 Image Description: {result['image_description']}")
    print(f"📌 Category: {result['category']}")
    print(f"🔗 URL: {result['url']}\n")



--- Image + Text Search Results ---
🔹 Model: Smart Bicycle Tail Light & Alarm System
📖 Chunk Text: Smart Bicycle Tail Light & Alarm System. This is about application Smart Bicycle Tail Light & Alarm System: This smart bicycle tail light and alarm system allows users to select different blinking patterns for various riding situations, such as stops, braking, and emergencies. Users can also trigger a horn as an alarm system and theft deterrent.
🖼 Image Path: /content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/diagrams/Smart_Bicycle_Tail_Light___Alarm_System.png
📝 Image Description: This image is a winning combination for application Smart Bicycle Tail Light & Alarm System.
📌 Category: https://www.renesas.com/en/applications/automotive/vehicle-control
🔗 URL: https://www.renesas.com/en/applications/automotive/vehicle-control/smart-bicycle-tail-light-alarm-system

🔹 Model: Smart Bicycle Tail Light & Alarm System
📖 Chunk Text: Smart Bicycle Tail Light & Ala

In [9]:
query_text = "Show me a diagram related to ADAS safety."

# Encode query text
query_embedding = text_model.encode(query_text).tolist()

# Query ChromaDB
results = image_text_collection.query(
    query_embeddings=[query_embedding],
    n_results=5
)

# Extract image paths
retrieved_images = [result["image_path"] for result in results["metadatas"][0] if result["image_path"]]

# Display Retrieved Image Paths
print("\nRetrieved Image Paths from ChromaDB:")
for img in retrieved_images:
    print(img)



Retrieved Image Paths from ChromaDB:
/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/diagrams/Smart_Bicycle_Tail_Light___Alarm_System.png
/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/diagrams/Smart_Bicycle_Tail_Light___Alarm_System.png
/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/diagrams/Smart_Bicycle_Tail_Light___Alarm_System.png
/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/diagrams/Smart_Bicycle_Tail_Light___Alarm_System.png
/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/diagrams/Smart_Bicycle_Tail_Light___Alarm_System.png


In [10]:
from PIL import Image

# Load and show the first retrieved image
if retrieved_images:
    image = Image.open(retrieved_images[0])  # Load the first image
    image.show()  # Display in Notebook


In [11]:
import base64
import io

def encode_image_base64(image_path):
    """Convert image to Base64 format."""
    with open(image_path, "rb") as image_file:
        return base64.b64encode(image_file.read()).decode("utf-8")

# Convert the first retrieved image
if retrieved_images:
    image_base64 = encode_image_base64(retrieved_images[0])
    print("Base64 Image String:", image_base64[:100])  # Print only the first 100 characters


Base64 Image String: iVBORw0KGgoAAAANSUhEUgAAAyAAAAJjCAYAAAD0y3UeAAAABmJLR0QA/wD/AP+gvaeTAAAgAElEQVR4nOzdd3hb5fn/8bdkyXvG


In [12]:
# Get all stored documents in ChromaDB
documents = image_text_collection.get()

# Count unique chunk texts
unique_chunks = set(doc["chunk_text"] for doc in documents["metadatas"])

print(f"Total Chunks in ChromaDB: {len(documents['ids'])}")
print(f"Unique Chunks in ChromaDB: {len(unique_chunks)}")

# If unique chunks are much lower than total, duplicates exist
if len(unique_chunks) < len(documents["ids"]):
    print("⚠️ Warning: Duplicate entries detected in ChromaDB!")


Total Chunks in ChromaDB: 257
Unique Chunks in ChromaDB: 257


In [13]:
# Load chunked data (Assuming it's a JSON list)
import json

file_path = "/content/drive/MyDrive/Advanced_RAG_Pipeline_On_Website/New_Final_winning_models_data/final_semantic_chunked_models.json"

# Load JSON
with open(file_path, "r", encoding="utf-8") as file:
    chunked_data = json.load(file)

# Count unique chunks
unique_chunks = set()
duplicate_chunks = []

for entry in chunked_data:
    chunk_text = entry["chunk_text"].strip()

    if chunk_text in unique_chunks:
        duplicate_chunks.append(chunk_text)
    else:
        unique_chunks.add(chunk_text)

print(f"Total Chunks: {len(chunked_data)}")
print(f"Unique Chunks: {len(unique_chunks)}")
print(f"Duplicate Chunks: {len(duplicate_chunks)}")

# Show some duplicates
if duplicate_chunks:
    print("\n⚠️ Duplicate Chunk Examples:")
    for chunk in duplicate_chunks[:5]:  # Show first 5 duplicates
        print(f"- {chunk[:200]}...\n")  # Print first 200 chars


Total Chunks: 257
Unique Chunks: 257
Duplicate Chunks: 0
